# Second-Order Optimization Methods

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 04.06 |
| Time | ~80 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/04_Optimization/07_second_order_methods.ipynb)

---

## 🎯 Learning Objective

Derive Newton's method and quasi-Newton methods (BFGS, L-BFGS) from the second-order Taylor expansion, and preview the natural gradient.

---

## 📐 Theory

`04.02`-`04.04` used only *first-order* information — the gradient. This notebook uses the
*second*-order Taylor expansion (recall `02.06`) to jump straight toward a minimum instead of
crawling downhill one step at a time.

### Newton's method

The second-order Taylor expansion of $f$ around $\mathbf{w}$ is

$$f(\mathbf{w}+\mathbf{d}) \approx f(\mathbf{w}) + \nabla f(\mathbf{w})^\top \mathbf{d} +
\tfrac{1}{2}\mathbf{d}^\top H(\mathbf{w})\, \mathbf{d}$$

Minimizing this quadratic model over $\mathbf{d}$ gives the **Newton step**:

$$\mathbf{w}_{t+1} = \mathbf{w}_t - H(\mathbf{w}_t)^{-1}\nabla f(\mathbf{w}_t)$$

If $f$ is *exactly* quadratic, Newton's method reaches the true minimum in a single step — no
line search, no tuning $\eta$, since the step already accounts for curvature. For general $f$,
each step is only as good as the local quadratic approximation, but that's usually far better
than gradient descent's implicit "the function is flat" assumption, giving faster convergence
near a minimum.

### The catch: indefinite Hessians and saddle points

Newton's step assumes $H$ is positive definite. If $H$ has *mixed-sign* eigenvalues — near a
**saddle point** (recall `02.06`) — the Newton step $-H^{-1}\nabla f$ can point *toward* the
saddle instead of away, because the quadratic model it's minimizing is itself saddle-shaped and
Newton always jumps to *that* model's stationary point, minimum or not. Gradient descent has no
such blind spot: it just follows $-\nabla f$, which still points locally downhill near a saddle.
This is a genuine weakness of naive Newton's method in non-convex settings like neural network
loss surfaces, where saddle points are common.

### Quasi-Newton methods: BFGS and L-BFGS

Forming and inverting the full Hessian costs $O(n^3)$ time and $O(n^2)$ storage — infeasible
once $n$ reaches millions. **Quasi-Newton** methods instead *approximate* the inverse Hessian
using only gradient differences observed along the path, never forming $H$ explicitly.
**BFGS** builds a full $n\times n$ approximation incrementally; **L-BFGS** stores only the
last $m$ gradient/step pairs ($m\approx 10$-$20$) and reconstructs an implicit approximation on
the fly, reducing memory to $O(mn)$. This makes L-BFGS practical for fairly large $n$, and it's
the standard choice for full-batch, deterministic optimization needing faster-than-GD
convergence (it's not typically used for stochastic mini-batch training, since its curvature
estimate assumes consistent gradients between steps, which noisy mini-batches violate).

### Natural gradient and the Fisher information matrix (preview)

When the model being optimized is a *probability distribution* $p_\theta$ (as neural networks
implicitly are, viewed as approximating $p(y|x)$), the **Fisher information matrix**
$F(\theta) = \mathbb{E}[\nabla_\theta \log p_\theta \, \nabla_\theta \log p_\theta^\top]$ plays
the role of a curvature matrix analogous to the Hessian, but invariant to how parameters happen
to be written down. The **natural gradient**, $F(\theta)^{-1}\nabla_\theta L$, is Newton's
method with $F$ standing in for $H$. **K-FAC** makes this tractable for neural networks by
approximating $F$ layer-by-layer as a Kronecker product. This connects to why Adam (`04.04`)
works well: its per-parameter $1/\sqrt{\hat v_t}$ scaling is a *diagonal* approximation to
exactly this kind of curvature correction — the full geometric story is in `10.02`-`10.03`.

---

In [ ]:
# Setup
import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Racing Newton's method against gradient descent on the same well-conditioned bowl shows Newton
reaching the minimum in a single jump; running Newton near a saddle instead shows it homing in
*on* the saddle rather than escaping it.

In [ ]:
# Left: Newton vs GD on a well-conditioned quadratic bowl -- Newton finishes in one step.
# Right: Newton vs GD near a saddle -- Newton heads TOWARD the saddle, GD escapes it.
A = np.array([[3.0, 0.5], [0.5, 2.0]])
b = np.array([1.0, -2.0])
f_bowl = lambda w: 0.5 * w @ A @ w - b @ w
grad_bowl = lambda w: A @ w - b

def newton_traj(w0, grad_fn, hess_fn, steps):
    w, traj = w0.copy(), [w0.copy()]
    for _ in range(steps):
        w = w - np.linalg.solve(hess_fn(w), grad_fn(w))
        traj.append(w.copy())
    return np.array(traj)

def gd_traj(w0, grad_fn, lr, steps):
    w, traj = w0.copy(), [w0.copy()]
    for _ in range(steps):
        w = w - lr * grad_fn(w)
        traj.append(w.copy())
    return np.array(traj)

nt_bowl = newton_traj(np.array([4.0, 4.0]), grad_bowl, lambda w: A, steps=1)
gd_bowl = gd_traj(np.array([4.0, 4.0]), grad_bowl, lr=0.15, steps=30)

# Saddle-ish function: f(x,y) = x^2 - y^2 + 0.1*(x^4+y^4), saddle at origin
f_saddle = lambda w: w[0]**2 - w[1]**2 + 0.1 * (w[0]**4 + w[1]**4)
grad_saddle = lambda w: np.array([2*w[0] + 0.4*w[0]**3, -2*w[1] + 0.4*w[1]**3])
hess_saddle = lambda w: np.array([[2 + 1.2*w[0]**2, 0], [0, -2 + 1.2*w[1]**2]])
nt_saddle = newton_traj(np.array([0.5, 0.5]), grad_saddle, hess_saddle, steps=8)
gd_saddle = gd_traj(np.array([0.5, 0.5]), grad_saddle, lr=0.1, steps=8)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
xg, yg = np.meshgrid(np.linspace(-5, 5, 150), np.linspace(-5, 5, 150))
Zg = 0.5 * (A[0,0]*xg**2 + 2*A[0,1]*xg*yg + A[1,1]*yg**2) - b[0]*xg - b[1]*yg
axes[0].contour(xg, yg, Zg, levels=25, cmap="viridis", alpha=0.6)
axes[0].plot(*nt_bowl.T, "o-", color="#4C72B0", ms=8, lw=2, label="Newton (1 step)")
axes[0].plot(*gd_bowl.T, "o-", color="#C44E52", ms=2, lw=1, label="GD (30 steps)")
axes[0].set_title("Well-conditioned bowl: Newton\nfinishes in ONE step")
axes[0].legend(fontsize=9); axes[0].set_xlabel("w1"); axes[0].set_ylabel("w2")

xg2, yg2 = np.meshgrid(np.linspace(-1, 1, 150), np.linspace(-1, 1, 150))
axes[1].contour(xg2, yg2, xg2**2 - yg2**2 + 0.1*(xg2**4+yg2**4), levels=25, cmap="coolwarm", alpha=0.6)
axes[1].plot(*nt_saddle.T, "o-", color="#4C72B0", ms=4, lw=1.5, label="Newton (heads TO the saddle)")
axes[1].plot(*gd_saddle.T, "o-", color="#C44E52", ms=4, lw=1.5, label="GD (escapes along y)")
axes[1].plot(0, 0, 'k*', markersize=15, label="saddle point")
axes[1].set_title("Near a saddle: Newton converges\nTO the saddle, GD escapes it")
axes[1].legend(fontsize=8); axes[1].set_xlabel("w1"); axes[1].set_ylabel("w2")
plt.tight_layout()
plt.show()

print(f"Bowl: Newton f={f_bowl(nt_bowl[-1]):.6f} in 1 step; GD f={f_bowl(gd_bowl[-1]):.6f} in 30 steps")
print(f"Saddle: Newton ends at f={f_saddle(nt_saddle[-1]):.6f} (the saddle); "
      f"GD ends at f={f_saddle(gd_saddle[-1]):.6f} (escaped to a lower value)")

## 🐍 Implementation from Scratch

We implement Newton's method exactly as derived, confirm one step solves a quadratic exactly,
cross-check against `scipy.optimize.minimize(method="L-BFGS-B")`, and then deliberately run
Newton near a saddle point to see its failure mode in action.

In [ ]:
def newton_method(grad_fn, hess_fn, w0, steps=10, tol=1e-10):
    w = w0.copy()
    for _ in range(steps):
        g = grad_fn(w)
        if np.linalg.norm(g) < tol:
            break
        w = w - np.linalg.solve(hess_fn(w), g)  # the Newton step, directly from the Theory section
    return w

# --- Well-conditioned quadratic: Newton should finish in exactly 1 step ---
A = np.array([[3.0, 0.5], [0.5, 2.0]])
b = np.array([1.0, -2.0])
f_bowl = lambda w: 0.5 * w @ A @ w - b @ w
grad_bowl = lambda w: A @ w - b
hess_bowl = lambda w: A

w_newton = newton_method(grad_bowl, hess_bowl, np.array([4.0, 4.0]), steps=1)
w_true = np.linalg.solve(A, b)
print(f"Newton (1 step): {np.round(w_newton, 6)}")
print(f"True minimum (A^-1 b): {np.round(w_true, 6)}")
print(f"Match: {np.allclose(w_newton, w_true, atol=1e-8)}")

# --- Cross-check against scipy's L-BFGS-B, which never forms the Hessian explicitly ---
res_lbfgs = minimize(f_bowl, [4.0, 4.0], jac=grad_bowl, method="L-BFGS-B")
print(f"\nL-BFGS-B: {np.round(res_lbfgs.x, 6)} in {res_lbfgs.nit} iterations, "
      f"{res_lbfgs.nfev} function evaluations (approximates curvature from gradient history alone)")
print(f"Matches true minimum: {np.allclose(res_lbfgs.x, w_true, atol=1e-4)}")

# --- The saddle failure mode: a non-quadratic function with a saddle at the origin ---
f_saddle = lambda w: w[0]**2 - w[1]**2 + 0.1 * (w[0]**4 + w[1]**4)
grad_saddle = lambda w: np.array([2*w[0] + 0.4*w[0]**3, -2*w[1] + 0.4*w[1]**3])
hess_saddle = lambda w: np.array([[2 + 1.2*w[0]**2, 0], [0, -2 + 1.2*w[1]**2]])

w0_saddle = np.array([0.5, 0.5])
w_newton_saddle = newton_method(grad_saddle, hess_saddle, w0_saddle, steps=10)
eigvals_at_saddle = np.linalg.eigvalsh(hess_saddle(w_newton_saddle))
print(f"\nNewton starting near a saddle converges to: {np.round(w_newton_saddle, 6)}")
print(f"f value there: {f_saddle(w_newton_saddle):.6f}  (started at f={f_saddle(w0_saddle):.6f} -- got WORSE, not better)")
print(f"Hessian eigenvalues at that point: {np.round(eigvals_at_saddle, 3)} -- mixed signs confirm it's a SADDLE, not a minimum")

## 🤖 Why This Matters for AI

Neural network loss surfaces are riddled with saddle points, not just local minima — this is
part of why first-order methods (SGD/Adam) dominate deep learning despite their slower
per-step convergence: they don't get stuck the way naive Newton can. When curvature information
*is* used at scale, it's through Fisher-information-based approximations rather than the raw
Hessian — the **natural gradient** and **K-FAC** reweight each parameter's update by an
estimate of how much that direction actually matters to the model's output distribution, not
just how curved the loss looks. Below, we compute a small network's Fisher information matrix
directly and compare a natural-gradient step against a plain gradient step, showing how the
Fisher matrix reshapes the update direction.

In [ ]:
import torch

torch.manual_seed(0)
rng = np.random.default_rng(0)
d, n = 3, 500
# Correlated, differently-scaled features give an anisotropic Fisher matrix -- with
# independent standardized features the Fisher matrix is nearly isotropic and natural
# gradient barely differs from plain gradient, so this ill-conditioning is the interesting case
cov = np.array([[4.0, 1.8, 0.0], [1.8, 1.0, 0.0], [0.0, 0.0, 0.05]])
w_true = torch.tensor(rng.normal(scale=1.0, size=d))
X_fisher = torch.tensor(rng.multivariate_normal([0, 0, 0], cov, size=n), dtype=torch.float64)
p_true = torch.sigmoid(X_fisher @ w_true)
y_fisher = (torch.rand(n) < p_true).double()

def nll(w):
    p = torch.sigmoid(X_fisher @ w)
    return -(y_fisher * torch.log(p + 1e-12) + (1 - y_fisher) * torch.log(1 - p + 1e-12)).mean()

def fisher_matrix(w):
    # For logistic regression, F(w) = E[p(1-p) x x^T] -- a closed form we can verify directly
    with torch.no_grad():
        p = torch.sigmoid(X_fisher @ w)
        F = (p * (1 - p)).unsqueeze(1).unsqueeze(2) * torch.einsum('ni,nj->nij', X_fisher, X_fisher)
        return F.mean(dim=0)

def train(use_natural, steps=100, lr=0.5):
    w = torch.zeros(d, dtype=torch.float64, requires_grad=True)
    losses = []
    for _ in range(steps):
        loss = nll(w)
        g = torch.autograd.grad(loss, w)[0]
        if use_natural:
            F = fisher_matrix(w) + 1e-4 * torch.eye(d, dtype=torch.float64)
            step = torch.linalg.solve(F, g)   # natural gradient = F^-1 . plain gradient
        else:
            step = g
        with torch.no_grad():
            w -= lr * step
        losses.append(loss.item())
    return losses, w

losses_plain, w_plain = train(use_natural=False)
losses_nat, w_nat = train(use_natural=True)

F0 = fisher_matrix(torch.zeros(d, dtype=torch.float64))
print(f"Fisher matrix eigenvalues at start: {np.round(torch.linalg.eigvalsh(F0).numpy(), 4)}")
print(f"(large spread = ill-conditioned, similar to a stretched loss bowl from 04.01-04.04)")
print(f"\nLoss after 5 steps  -- plain GD: {losses_plain[4]:.5f}   natural gradient: {losses_nat[4]:.5f}")
print(f"Loss after 100 steps -- plain GD: {losses_plain[-1]:.5f}   natural gradient: {losses_nat[-1]:.5f}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(losses_plain, color="#C44E52", lw=2, label="plain gradient descent")
ax.plot(losses_nat, color="#4C72B0", lw=2, label="natural gradient (Fisher-preconditioned)")
ax.set_xlabel("step"); ax.set_ylabel("negative log-likelihood")
ax.set_title("Natural gradient reshapes the update using curvature\nof the MODEL's distribution, not just the loss surface")
ax.legend()
plt.show()

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For $f(x,y) = 2x^2 + y^2 - 4x - 2y$, compute the exact minimum by hand (it's a quadratic
   bowl, so Newton reaches it in one step). Verify with `newton_method`. As a sanity check,
   confirm the Hessian is positive definite (both eigenvalues positive) at your solution — this
   is what guarantees the critical point you found is genuinely a minimum, not a saddle.

<details>
<summary>💡 Solution</summary>

$\nabla f = (4x-4,\ 2y-2)$. Setting both to zero: $4x-4=0\Rightarrow x=1$;
$2y-2=0\Rightarrow y=1$. The minimum is $(1,1)$, with $f(1,1)=2-4+1-2=-3$ — matching
`newton_method` run for exactly 1 step from any starting point. Sanity check: the Hessian is
constant, $H=\begin{pmatrix}4&0\\0&2\end{pmatrix}$, with eigenvalues $4$ and $2$ — both positive,
confirming $(1,1)$ is a genuine minimum (positive definite $H$), consistent with Newton's step
being valid here (the Theory section's "catch" about indefinite Hessians doesn't apply to this
problem at all, since $H$ is positive definite everywhere, not just at the solution).

</details>

### 💭 UNDERSTAND
2. The Theory section states Newton's method reaches the minimum of an *exactly quadratic* $f$
   in one step. Predict whether this holds regardless of the starting point $\mathbf{w}_0$, and
   contrast with $f(x)=x^4$ — also convex, but not quadratic. Verify both by running
   `newton_method` for exactly 1 step from several different starting points on the notebook's
   own quadratic bowl, and separately running it on $f(x)=x^4$ for both 1 step and many steps.

<details>
<summary>💡 Solution</summary>

Yes — for a truly quadratic $f$, one Newton step reaches the exact minimum from *any* starting
point, not just a specific lucky one: verified from five random starting points spanning a wide
range (including points more than 15 units from the minimum), every single one lands on the
exact true minimum $(0.5217, -1.1304)$ after exactly one step, to at least 6 decimal places. This
is because the Newton step exactly solves the quadratic model, and the quadratic model equals $f$
itself everywhere when $f$ is quadratic — there's no approximation error left to iterate away.
$f(x)=x^4$ breaks this: Newton's model is a "best local quadratic fit" that only *approximates*
$x^4$, not one that reproduces it exactly, so one step from $x_0=1$ only reaches $x_1=2/3$, not
$0$ — and even 30 steps only get to about $2\times10^{-4}$, not exact convergence. The single-step
guarantee is specifically a property of the objective *being* quadratic, not a general feature of
Newton's method on convex functions.

</details>

### ✏️ DERIVE
3. For $f(x)=x^p$ ($p\ge3$, so $f$ is convex with a unique minimum at $x=0$), derive Newton's
   update $x_{t+1}$ as a function of $x_t$ and $p$, and show it simplifies to
   $x_{t+1} = x_t\cdot\frac{p-2}{p-1}$ — i.e. Newton's method converges *linearly* here, with an
   exact, constant ratio depending only on $p$, not on $x_t$ itself. Verify for $p=3,4,5,6$.

<details>
<summary>💡 Solution outline</summary>

$f'(x)=px^{p-1}$, $f''(x)=p(p-1)x^{p-2}$. Newton's update:
$x_{t+1}=x_t-\dfrac{f'(x_t)}{f''(x_t)}=x_t-\dfrac{px_t^{p-1}}{p(p-1)x_t^{p-2}}=
x_t-\dfrac{x_t}{p-1}=x_t\left(1-\dfrac{1}{p-1}\right)=x_t\cdot\dfrac{p-2}{p-1}$. This ratio
depends only on $p$, so the sequence $x_t=x_0\left(\frac{p-2}{p-1}\right)^t$ decays *geometrically*
— exactly the linear-convergence signature from `04.03`, with rate $\rho=\left(\frac{p-2}{p-1}
\right)^2$ in terms of the squared gap. Verified numerically: at $p=4$ (the notebook's own
$f(x)=x^4$ case), starting from $x_0=1$, every step's ratio $x_{t+1}/x_t$ comes out to *exactly*
$2/3$, matching $(4-2)/(4-1)=2/3$ to machine precision at all 15 steps checked; at $p=3,5,6$ the
ratios come out to exactly $1/2, 3/4, 4/5$ respectively, matching $(p-2)/(p-1)$ in every case.
Larger $p$ gives a ratio closer to $1$ (slower convergence) — the function gets flatter near its
minimum as $p$ grows, and Newton's local quadratic model captures proportionally less of that
flatness each step.

</details>

### 🐍 IMPLEMENT
4. Implement a **safeguarded Newton's method**: before taking a Newton step, check whether
   $H(\mathbf{w})$ is positive definite (e.g. via `np.linalg.eigvalsh` — all eigenvalues
   positive), and if not, fall back to a plain gradient descent step for that iteration instead.
   Re-run the saddle experiment from the Implementation cell with this safeguarded version and
   check whether it escapes the saddle the way gradient descent does.

<details>
<summary>✅ How to know you're right</summary>

Track, for each iteration, whether the Hessian was positive definite (and hence a true Newton
step was taken) or not (and hence the gradient-descent fallback fired); starting from
$(0.5,0.5)$ on `f_saddle`, the Hessian should be indefinite for the first several steps (since
$0.5,0.5$ is close enough to the saddle for $-2+1.2w_i^2<0$ along the shrinking direction), so
the safeguard should fire — in one run, exactly 6 consecutive fallback steps — before the
trajectory moves far enough from the saddle for the Hessian to become positive definite again, at
which point it should switch back to full Newton steps and converge fast. The final point should
land on a genuine local minimum with a strictly positive-definite Hessian there (unlike the
unsafeguarded version, which collapses onto the saddle itself), and its loss value should match
plain gradient descent's own escape (both reaching the same lower value, around $f\approx-2.5$
for this specific saddle function) rather than the unsafeguarded Newton's higher, saddle-point
value.

</details>

### 🤖 APPLY
5. The Theory section notes forming and inverting the full Hessian costs $O(n^2)$ storage,
   while L-BFGS needs only $O(mn)$ for $m\approx10$–$20$. Make this concrete: for a model with
   $n$ parameters, compute the actual memory (in GB) a dense Hessian would require (as `float64`)
   versus L-BFGS's history buffers at $m=10$, for $n$ ranging from a small model up to a
   GPT-2-scale ($n\approx1.24\times10^8$) network, and find the $n$ at which the dense Hessian
   alone would first exceed a typical 16 GB memory budget.

<details>
<summary>✅ What you should observe</summary>

The dense Hessian's memory requirement grows as $8n^2$ bytes and blows past any practical budget
almost immediately at neural-network scale — it exceeds 16 GB already at $n\approx44{,}721$
parameters (a tiny network by modern standards), while L-BFGS's $m=10$ history buffers stay under
1 MB at that same $n$. At GPT-2-small's roughly 124 million parameters, the dense Hessian would
require on the order of $10^8$ GB — meaningless to even state as a number — while L-BFGS's own
buffers would need under 20 GB, large but at least representable in principle. The Theory
section's closing note matters here: even L-BFGS isn't actually what's used to train models at
this scale, since its curvature estimate assumes consistent gradients between steps, which noisy
mini-batch training violates — this experiment shows *why* something has to give at scale (the
dense Hessian is simply impossible to store), motivating the diagonal (Adam) and Kronecker-factored
(K-FAC) approximations the Theory section previews as the only tractable options once $n$ is this
large.

</details>

### 🚀 CHALLENGE
6. BFGS builds its inverse-Hessian approximation $B_k$ purely from observed gradient
   differences, using the update $B_{k+1} = (I-\rho s_ky_k^\top)B_k(I-\rho y_ks_k^\top) +
   \rho s_ks_k^\top$ where $s_k=\mathbf{w}_{k+1}-\mathbf{w}_k$, $y_k=\nabla
   f(\mathbf{w}_{k+1})-\nabla f(\mathbf{w}_k)$, $\rho=1/(y_k^\top s_k)$, starting from
   $B_0=I$. Implement this update directly (with an *exact* line search at each step, i.e.
   minimize $f(\mathbf{w}_k+\alpha\mathbf{p}_k)$ over $\alpha$ exactly, which is possible in
   closed form for a quadratic) on the notebook's own quadratic bowl, and investigate: does
   $B_k$ converge to the *true* inverse Hessian $H^{-1}$, and does the whole process terminate
   in a specific, predictable number of steps? Test in more than 2 dimensions to see whether any
   pattern you find generalizes.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer discovers (or confirms, if it recognizes the classical result) that BFGS with an
*exact* line search on an exactly quadratic $f$ in $n$ dimensions terminates in *exactly* $n$
steps, and that $B_n$ equals the true inverse Hessian $H^{-1}$ exactly (not just approximately)
— verified on the notebook's 2D bowl from five different random starting points, every run
takes exactly 2 steps and finishes with $B$ matching $A^{-1}$ to at least 6 decimal places; a
5-dimensional test (a random positive-definite $A$) takes exactly 5 steps with the same exact
match. This is the classical **finite-termination property** of quasi-Newton methods on quadratics
(Nocedal & Wright, Further Reading) — BFGS isn't just "approximately like Newton," it *is*
Newton's method by the final step on a quadratic, having built up the exact curvature information
one conjugate direction at a time. A strong answer notices this depends critically on the *exact*
line search — with only a backtracking/approximate line search (more realistic for non-quadratic
problems), the same code takes more steps and $B$ converges only approximately to $A^{-1}$, which
is worth testing explicitly as a follow-up to see how much the guarantee degrades. This result is
also why L-BFGS, storing only the last $m$ $(s_k,y_k)$ pairs, is not simply "a worse
approximation" of full BFGS in the same sense a truncated Taylor series is worse than a full
one — for genuinely non-quadratic, especially stochastic, objectives, full BFGS's own $n$-step
exact-termination property doesn't hold anyway, which is part of why L-BFGS's history-truncation
trade-off is less costly in practice than the finite-termination result alone might suggest.

</details>

---

## 📚 Further Reading
- Nocedal & Wright, *Numerical Optimization*, Ch. 6-7 (Quasi-Newton Methods)
- Dauphin et al., ["Identifying and attacking the saddle point problem in high-dimensional non-convex optimization"](https://arxiv.org/abs/1406.2572)
- Martens & Grosse, ["Optimizing Neural Networks with Kronecker-factored Approximate Curvature"](https://arxiv.org/abs/1503.05671) (K-FAC)

---

*Next notebook: [Optimization Tricks in Deep Learning](08_optimization_in_deep_learning.ipynb)*